# Bab 06 · Berkas, Pengodean, dan Penanganan Galat

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Membaca teks UTF-8 dan CSV yang mengandung koma dalam nilai.
- Menangani galat yang memang diantisipasi.
- Memastikan sumber daya dibersihkan melalui context manager.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 9

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 9

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 9

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 9

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 9

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 9

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 9

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 9

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 9

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import csv
import json
from contextlib import contextmanager

## [BACA] Konsep inti

Teks disimpan sebagai byte dengan pengodean tertentu. Sebutkan UTF-8 ketika membaca atau menulis. CSV memiliki aturan kutip, sehingga `split(",")` tidak cukup. Tangkap exception spesifik; `except` terlalu luas dapat menyembunyikan bug. Pengujian berkas di sini memakai folder sementara.

## [DUGA] Prediksi sebelum eksekusi

Pembacaan mana yang menghasilkan teks benar? Mengapa jumlah potongan berbeda?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
teks = "Kafé, rasa vanila"
data = teks.encode("utf-8")
print(data.decode("utf-8"))
print(data.decode("latin-1"))
contoh = '"Kue, premium",85000'
print(contoh.split(","))
print(next(csv.reader([contoh])))

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Membaca bilangan dengan galat spesifik

**[ISI KODE]**

Buat `baca_angka(jalur)`: baca teks UTF-8, hilangkan spasi tepi, lalu ubah menjadi `int`. Kembalikan `None` hanya untuk berkas tidak ada atau teks bukan bilangan bulat. Galat lain harus diteruskan.

> Petunjuk: Batasi klausa `except` pada dua jenis exception yang disebutkan.

In [ ]:
# [ISI KODE]
def baca_angka(jalur):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    with TemporaryDirectory() as folder:
        p = Path(folder) / "angka.txt"
        p.write_text("  -12  ", encoding="utf-8")
        sama(baca_angka(p), -12)
        p.write_text("dua", encoding="utf-8")
        sama(baca_angka(p), None)
        sama(baca_angka(Path(folder) / "hilang.txt"), None)
        harus_galat(IsADirectoryError, lambda: baca_angka(Path(folder)))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · CSV dan laporan baris rusak

**[ISI KODE]**

Buat `baca_harga(jalur)` untuk CSV UTF-8 berheader `produk,harga`. Kembalikan `(daftar_valid, nomor_baris_rusak)`. Setiap hasil valid berupa dict `{"produk": teks, "harga": int}`. Harga negatif atau bukan int dianggap rusak. Nama produk tidak kosong, jumlah kolom selalu tepat, dan tidak ada newline di dalam kolom. Header adalah baris 1; koma dalam nama harus tetap terbaca.

> Petunjuk: Gunakan `csv.DictReader` dan `enumerate(..., start=2)`.

In [ ]:
# [ISI KODE]
def baca_harga(jalur):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    with TemporaryDirectory() as folder:
        p = Path(folder) / "harga.csv"
        p.write_text(
            'produk,harga\n"Kue, premium",85000\nLapis,salah\nRoti,-1\nKafé,0\n',
            encoding="utf-8",
        )
        valid, rusak = baca_harga(p)
        sama(
            valid,
            [
                {"produk": "Kue, premium", "harga": 85000},
                {"produk": "Kafé", "harga": 0},
            ],
        )
        sama(rusak, [3, 4])
        p.write_text("produk,harga\n", encoding="utf-8")
        sama(baca_harga(p), ([], []))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · JSON yang dapat dibaca kembali

**[ISI KODE]**

Buat `simpan_json(data, jalur)`: simpan dict/list JSON-compatible sebagai UTF-8 dengan karakter non-ASCII tetap tertulis langsung (`ensure_ascii=False`), lalu kembalikan `Path` berkas tersebut.

> Petunjuk: Gunakan modul `json`, bukan mengubah dict menjadi `str`.

In [ ]:
# [ISI KODE]
def simpan_json(data, jalur):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    with TemporaryDirectory() as folder:
        data = {"nama": "Kafé", "stok": [2, 0], "aktif": True}
        p = simpan_json(data, Path(folder) / "data.json")
        assert isinstance(p, Path), "Kembalikan Path."
        teks = p.read_text(encoding="utf-8")
        assert "Kafé" in teks, "Karakter non-ASCII harus tertulis langsung."
        sama(json.loads(teks), data)

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Context manager yang membersihkan berkas

**[ISI KODE]**

Lengkapi context manager `berkas_sementara(folder)`. Buat `uji-sementara.txt` baru di folder yang disediakan, serahkan objek `Path` melalui `yield`, dan hapus saat keluar, termasuk jika terjadi exception. Jangan menimpa berkas yang sudah ada: munculkan `FileExistsError`. Jangan menelan exception dari blok pemanggil.

> Petunjuk: Letakkan pembersihan di `finally`, setelah berhasil membuat berkas baru.

In [ ]:
# [ISI KODE]
@contextmanager
def berkas_sementara(folder):
    raise BelumDiisi()
    yield  # menjaga bentuk generator; lengkapi implementasinya

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    with TemporaryDirectory() as folder:
        with berkas_sementara(folder) as p:
            assert p.exists(), "Berkas harus ada selama blok berjalan."
            p.write_text("isi", encoding="utf-8")
        assert not p.exists(), "Berkas harus dihapus setelah blok selesai."
        try:
            with berkas_sementara(folder) as p2:
                raise RuntimeError("galat sengaja")
        except RuntimeError:
            pass
        else:
            raise AssertionError("Exception pemanggil tidak boleh ditelan.")
        assert (
            not p2.exists()
        ), "Pembersihan wajib terjadi ketika ada exception."
        p3 = Path(folder) / "uji-sementara.txt"
        p3.write_text("milik pengguna", encoding="utf-8")

        def bentrok():
            with berkas_sementara(folder):
                pass

        harus_galat(FileExistsError, bentrok)
        sama(p3.read_text(encoding="utf-8"), "milik pengguna")

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.